## Setup: Install Libraries

In [11]:
# ================== INSTALL ==================
!pip install ipywidgets folium geopy -q

## Imports

In [1]:
# ================== IMPORTS ==================
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from geopy.geocoders import Nominatim
import folium
import warnings
warnings.filterwarnings('ignore')

## Initialize Geolocator

In [13]:
# ================== GEOLOCATOR ==================
geolocator = Nominatim(user_agent="smart_uber_app")

## Load and Prepare Data

In [14]:
# ================== LOAD DATA ==================
df = pd.read_csv("amritsar_uber_dataset.csv")

df.rename(columns={
    'pickup_location': 'pickup_location',
    'drop_location': 'dropoff_location',
    'fare': 'fare_amount'
}, inplace=True)

df.dropna(inplace=True)

## Generate Location Coordinates

In [15]:
# ================== LOCATION DICT ==================
locations = list(set(df['pickup_location']).union(set(df['dropoff_location'])))

np.random.seed(42)
location_coords = {}

for loc in locations:
    location_coords[loc] = (
        31.6 + np.random.uniform(-0.1, 0.1),
        74.8 + np.random.uniform(-0.1, 0.1)
    )

## Encode Locations

In [19]:

# ================== ENCODING ==================
le_pickup = LabelEncoder()
le_dropoff = LabelEncoder()

df['pickup_code'] = le_pickup.fit_transform(df['pickup_location'])
df['dropoff_code'] = le_dropoff.fit_transform(df['dropoff_location'])

## Train Random Forest Model

In [20]:

# ================== MODEL ==================
import numpy as np

# Dynamically identify the passenger and hour columns from the dataset
passenger_col = [col for col in df.columns if 'pass' in col.lower()]
passenger_col = passenger_col[0] if passenger_col else None

hour_col = [col for col in df.columns if 'hour' in col.lower() or 'time' in col.lower()]
hour_col = hour_col[0] if hour_col else None

# Calculate distance_km using pickup and dropoff coordinates if not present
if 'distance_km' not in df.columns:
    if all(col in df.columns for col in ['pickup_lat', 'pickup_lng', 'drop_lat', 'drop_lng']):
        # Simple Euclidean approximation converted to km
        df['distance_km'] = ((df['drop_lat'] - df['pickup_lat'])**2 + (df['drop_lng'] - df['pickup_lng'])**2)**0.5 * 111
    else:
        df['distance_km'] = 5.0 # Fallback default

# Fallbacks if columns are not found
if passenger_col:
    df['passenger_count'] = df[passenger_col]
else:
    df['passenger_count'] = np.random.randint(1, 5, size=len(df))

# Convert categorical times of day into numeric representation
if hour_col:
    # Map strings like 'Morning', 'Afternoon', 'Evening', 'Night' to numeric values
    time_mapping = {
        'morning': 9,
        'afternoon': 14,
        'evening': 18,
        'night': 22
    }

    # Check if the values are strings or already numeric
    if df[hour_col].dtype == object:
        df['hour'] = df[hour_col].str.lower().map(time_mapping).fillna(12)
    else:
        df['hour'] = df[hour_col]
else:
    df['hour'] = np.random.randint(0, 24, size=len(df))

features = ['pickup_code', 'dropoff_code', 'distance_km', 'passenger_count', 'hour']
X = df[features]
y = df['fare_amount']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print("✅ Model Ready")
print("R2:", r2_score(y_test, model.predict(X_test)))

ValueError: could not convert string to float: '2024-01-26 09:10:00'

## Helper Functions: Geocoding and Suggestions

In [15]:


# ================== GEO FUNCTION ==================
def get_coordinates(place):
    try:
        loc = geolocator.geocode(place)
        if loc:
            return (loc.latitude, loc.longitude)
    except:
        return None
    return None

# ================== SUGGESTION FUNCTION ==================
def suggest_location(name):
    for loc in location_coords.keys():
        if name.lower() in loc.lower():
            return loc
    return None

## Helper Function: Get Location from UI Inputs

In [16]:
# ================== GET LOCATION ==================
def get_location(dropdown, textbox):
    if textbox.value.strip() != "":
        coords = get_coordinates(textbox.value)
        if coords:
            return textbox.value, coords
        else:
            suggestion = suggest_location(textbox.value)
            if suggestion:
                return suggestion, location_coords[suggestion]
            return None, None
    else:
        loc = dropdown.value
        return loc, location_coords[loc]

## Helper Function: Predict Fare

In [17]:
# ================== PREDICT FUNCTION ==================
def predict_fare(p_dd, p_txt, d_dd, d_txt, pax, hr, vehicle, bags_count):

    p_name, p_coord = get_location(p_dd, p_txt)
    d_name, d_coord = get_location(d_dd, d_txt)

    if p_coord is None or d_coord is None:
        return "❌ Location not found. Try nearby place."

    if p_name == d_name:
        return "❌ Pickup & Drop same hai"

    # Passenger and Bag Restrictions
    if vehicle == "Activa" or vehicle == "Bike":
        if pax > 1:
            return "❌ Activa/Bike allows only 1 passenger."
        if bags_count > 2:
            return "❌ Activa/Bike allows a maximum of 2 bags/items."
    elif vehicle == "Car":
        if pax > 4:
            return "❌ Car allows a maximum of 4 passengers."
        if bags_count > 5:
            return "❌ Car allows a maximum of 5 bags/items."

    dist = ((d_coord[0]-p_coord[0])**2 + (d_coord[1]-p_coord[1])**2)**0.5 * 111

    # Error handling for unknown locations in LabelEncoder
    try:
        p_code = le_pickup.transform([p_name])[0]
    except ValueError:
        return f"❌ Pickup location '{p_name}' not recognized by the model. Please select from the dropdown or use a recognized custom location."

    try:
        d_code = le_dropoff.transform([d_name])[0]
    except ValueError:
        return f"❌ Dropoff location '{d_name}' not recognized by the model. Please select from the dropdown or use a recognized custom location."

    # Wrap input data into a DataFrame matching feature names
    input_data = pd.DataFrame([[
        p_code, d_code, dist, pax, hr
    ]], columns=['pickup_code', 'dropoff_code', 'distance_km', 'passenger_count', 'hour'])

    fare = model.predict(input_data)[0]

    # 🚗 Vehicle Pricing and Bag Charges
    bag_charge_per_item = 20 # Example charge per extra bag
    if vehicle == "Activa":
        fare *= 0.6
        if bags_count > 0:
            fare += bags_count * bag_charge_per_item # Charge for each bag
    elif vehicle == "Bike":
        fare *= 0.8
        if bags_count > 0:
            fare += bags_count * bag_charge_per_item # Charge for each bag
    elif vehicle == "Car":
        fare *= 1.2
        if bags_count > 0:
            fare += bags_count * bag_charge_per_item # Charge for each bag

    return f"""
🚗 {p_name} → {d_name}
🛵 Vehicle: {vehicle}
👥 Passengers: {pax}
🎒 Bags/Items: {bags_count}
💰 Fare: ₹{fare:.0f}
📏 Distance: {dist:.2f} km
"""

## Helper Function: Show Map

In [18]:
# ================== MAP ==================
def show_map(p_coord, d_coord, p_name, d_name):
    m = folium.Map(location=[(p_coord[0]+d_coord[0])/2, (p_coord[1]+d_coord[1])/2], zoom_start=12)

    folium.Marker(p_coord, popup=p_name, icon=folium.Icon(color='green')).add_to(m)
    folium.Marker(d_coord, popup=d_name, icon=folium.Icon(color='red')).add_to(m)

    folium.PolyLine([p_coord, d_coord], color='blue').add_to(m)

    return m._repr_html_()

## UI Widgets

In [19]:
# ================== UI ==================
pickup_dd = widgets.Dropdown(options=list(location_coords.keys()), description='From:')
pickup_txt = widgets.Text(placeholder='Type location...', description='Custom:')

drop_dd = widgets.Dropdown(options=list(location_coords.keys()), description='To:')
drop_txt = widgets.Text(placeholder='Type location...', description='Custom:')

pax = widgets.BoundedIntText(value=1, min=1, max=4, description='Passengers:')
time = widgets.BoundedIntText(value=12, min=0, max=23, description='Hour:')
bags = widgets.BoundedIntText(value=0, min=0, max=5, description='Bags/Items:')

vehicle = widgets.Dropdown(options=['Activa', 'Bike', 'Car'], description='Vehicle:')

btn = widgets.Button(description="🚀 GET FARE", button_style='success')

out1 = widgets.Output()
out2 = widgets.Output()

## Button Click Handler

In [20]:
# ================== BUTTON ==================
def on_click(b):
    with out1:
        clear_output()
        result = predict_fare(pickup_dd, pickup_txt, drop_dd, drop_txt, pax.value, time.value, vehicle.value, bags.value)
        print(result)

    with out2:
        clear_output()
        p_name, p_coord = get_location(pickup_dd, pickup_txt)
        d_name, d_coord = get_location(drop_dd, drop_txt)

        if p_coord and d_coord:
            map_html = show_map(p_coord, d_coord, p_name, d_name)
            display(HTML(map_html)) # Display the map with found coordinates

btn.on_click(on_click)

In [21]:
# ================== STATE & ADVANCED HELPERS ==================
import datetime

# Global Ride History storage
ride_history_data = [
    {"Date": "2023-10-24 09:30", "Pickup": "Golden Temple", "Destination": "Amritsar Junction", "Vehicle": "Car", "Distance": "2.5 km", "Fare": "₹120", "Duration": "12 min", "Rating": "⭐⭐⭐⭐⭐"},
    {"Date": "2023-10-24 14:15", "Pickup": "Airport", "Destination": "Ranjit Avenue", "Vehicle": "Bike", "Distance": "11.2 km", "Fare": "₹160", "Duration": "22 min", "Rating": "⭐⭐⭐⭐"}
]

# User Ratings and Feedback lists
user_feedbacks = [
    {"Rating": 5, "Feedback": "Excellent service, very safe drive!", "Safety": "Safe"},
    {"Rating": 4, "Feedback": "On time and clean ride.", "Safety": "Safe"}
]

def get_average_rating():
    if not user_feedbacks: return 5.0
    return round(sum(f['Rating'] for f in user_feedbacks) / len(user_feedbacks), 1)

# Helper for Traffic, Peak Hours & Route Details
def analyze_ride_conditions(hour, dist):
    # Peak Hour Detection
    is_peak = False
    peak_reason = "Normal Hours"
    if (8 <= hour <= 11):
        is_peak = True
        peak_reason = "Morning Rush Peak Hour"
    elif (17 <= hour <= 21):
        is_peak = True
        peak_reason = "Evening Office/Market Peak Hour"

    # Traffic Level Estimation based on hours
    if is_peak:
        traffic = "High"
        traffic_mult = 1.3
        speed_kmh = 20
    elif (22 <= hour <= 23) or (0 <= hour <= 6):
        traffic = "Low"
        traffic_mult = 0.95
        speed_kmh = 45
    else:
        traffic = "Medium"
        traffic_mult = 1.1
        speed_kmh = 30

    # Travel Time Calculation
    est_time_mins = int((dist / speed_kmh) * 60) + 3 # base offset of 3 mins

    return is_peak, peak_reason, traffic, traffic_mult, est_time_mins

print("⚙️ Advanced helpers and global states successfully registered!")

⚙️ Advanced helpers and global states successfully registered!


In [22]:
# ================== METRICS & PLOT GENERATORS ==================
import matplotlib.pyplot as plt
import seaborn as sns
import io
import base64
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Precalculate Model Metrics
y_pred = model.predict(X_test)
r2_val = r2_score(y_test, y_pred)
mae_val = mean_absolute_error(y_test, y_pred)
rmse_val = np.sqrt(mean_squared_error(y_test, y_pred))

# Helper to encode matplotlib plots to base64 HTML string to embed smoothly in UI
def fig_to_html(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight', dpi=100)
    buf.seek(0)
    img_str = base64.b64encode(buf.read()).decode('utf-8')
    plt.close(fig)
    return f'<img src="data:image/png;base64,{img_str}" style="max-width:100%; border-radius:10px; box-shadow: 0 4px 6px rgba(0,0,0,0.1);"/>'

def generate_analytics_dashboard():
    # Plot 1: Feature Importance
    fig1, ax1 = plt.subplots(figsize=(6, 4.5))
    importances = model.feature_importances_
    indices = np.argsort(importances)
    colors = sns.color_palette("viridis", len(features))
    ax1.barh([features[i] for i in indices], importances[indices], color=colors)
    ax1.set_title("Random Forest Feature Importance", fontsize=12, fontweight='bold')
    ax1.set_xlabel("Relative Importance")
    fig1.tight_layout()
    img_features = fig_to_html(fig1)

    # Plot 2: Fare vs Distance Regression Trend
    fig2, ax2 = plt.subplots(figsize=(6, 4.5))
    sample_df = df.sample(min(1000, len(df)), random_state=42)
    sns.regplot(data=sample_df, x='distance_km', y='fare_amount', ax=ax2,
                scatter_kws={'alpha':0.4, 'color':'#2b5c8f'}, line_kws={'color':'red', 'lw':2})
    ax2.set_title("Distance vs Fare Distribution (Amritsar Uber)", fontsize=12, fontweight='bold')
    ax2.set_xlabel("Distance (km)")
    ax2.set_ylabel("Predicted Fare (₹)")
    fig2.tight_layout()
    img_trend = fig_to_html(fig2)

    # Plot 3: Time of Day Fare Breakdown
    fig3, ax3 = plt.subplots(figsize=(6, 4.5))
    sns.boxplot(data=df, x='time_of_day' if 'time_of_day' in df.columns else 'hour', y='fare_amount', ax=ax3, palette="coolwarm")
    ax3.set_title("Fare Range by Time of Day", fontsize=12, fontweight='bold')
    ax3.set_xlabel("Time Slot")
    ax3.set_ylabel("Fare (₹)")
    fig3.tight_layout()
    img_time = fig_to_html(fig3)

    # Plot 4: Passenger Count VS Fare
    fig4, ax4 = plt.subplots(figsize=(6, 4.5))
    sns.barplot(data=df, x='passenger_count', y='fare_amount', ax=ax4, palette="Blues_d")
    ax4.set_title("Average Fare by Passenger Capacity", fontsize=12, fontweight='bold')
    ax4.set_xlabel("Passenger Count")
    ax4.set_ylabel("Average Fare (₹)")
    fig4.tight_layout()
    img_pass = fig_to_html(fig4)

    return img_features, img_trend, img_time, img_pass

print("📈 Analytical dashboard plot engines established!")

📈 Analytical dashboard plot engines established!


In [31]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import datetime
import pandas as pd

# --- Operational Tab Outputs ---
out_predict = widgets.Output()
out_compare = widgets.Output()
out_safety = widgets.Output()

# --- Sleek Mobile App CSS ---
style_header = """
<style>
    .app-header {
        background: #09091a;
        color: #ffffff;
        padding: 15px;
        border-radius: 12px;
        text-align: center;
        margin-bottom: 20px;
        font-family: 'Segoe UI', Arial, sans-serif;
        box-shadow: 0 4px 15px rgba(0,0,0,0.15);
    }
    .card {
        border: 1px solid #e2e8f0;
        border-radius: 12px;
        padding: 18px;
        background: #ffffff;
        box-shadow: 0 4px 12px rgba(0,0,0,0.05);
        margin-bottom: 15px;
        font-family: 'Segoe UI', Arial, sans-serif;
    }
    .tbl {
        width: 100%;
        border-collapse: collapse;
        margin-top: 15px;
    }
    .tbl th, .tbl td {
        padding: 12px;
        text-align: left;
        border-bottom: 1px solid #edf2f7;
    }
    .tbl th {
        background-color: #f7fafc;
        color: #4a5568;
        font-weight: 600;
    }
</style>
"""

# ------------------ 1. BOOK RIDE TAB ------------------
p_dd = widgets.Dropdown(options=sorted(list(location_coords.keys())), description='Pickup:', layout={'width': '45%'}, value="Golden Temple")
p_txt = widgets.Text(placeholder='Or type custom address...', description='Custom:', layout={'width': '45%'});

default_drop = "Amritsar Airport (ATQ)" if "Amritsar Airport (ATQ)" in location_coords else list(location_coords.keys())[0]
d_dd = widgets.Dropdown(options=sorted(list(location_coords.keys())), description='Dropoff:', layout={'width': '45%'}, value=default_drop)
d_txt = widgets.Text(placeholder='Or type custom address...', description='Custom:', layout={'width': '45%'});

pax_widget = widgets.BoundedIntText(value=1, min=1, max=4, description='Passengers:', layout={'width': '45%'});
hr_widget = widgets.BoundedIntText(value=datetime.datetime.now().hour, min=0, max=23, description='Hour (0-23):', layout={'width': '45%'});
bags_widget = widgets.BoundedIntText(value=0, min=0, max=5, description='Bags/Items:', layout={'width': '45%'});
veh_widget = widgets.Dropdown(options=['Car', 'Bike', 'Activa', 'Auto Rickshaw'], description='Ride Type:', layout={'width': '45%'}, value="Car")

btn_predict = widgets.Button(description="🚗 BOOK / CALCULATE FARE", button_style='success', layout={'width': '92%', 'margin': '15px 0px', 'height': '40px'})
out_predict_box = widgets.Output()

def on_predict_clicked(b):
    with out_predict_box:
        clear_output()

        p_name, p_coord = get_location(p_dd, p_txt)
        d_name, d_coord = get_location(d_dd, d_txt)

        if not p_coord or not d_coord:
            display(HTML("<div class='card' style='color:red;'>❌ Location not found. Please spell correctly or pick from the list.</div>"))
            return

        dist = ((d_coord[0]-p_coord[0])**2 + (d_coord[1]-p_coord[1])**2)**0.5 * 111
        is_peak, peak_reason, traffic, traffic_mult, est_time = analyze_ride_conditions(hr_widget.value, dist)

        try:
            p_code = le_pickup.transform([p_name])[0]
            d_code = le_dropoff.transform([d_name])[0]
        except:
            display(HTML(f"<div class='card' style='color:red;'>❌ Endpoint location is out of service zone bounds.</div>"))
            return

        input_df = pd.DataFrame([[p_code, d_code, dist, pax_widget.value, hr_widget.value]],
                                columns=['pickup_code', 'dropoff_code', 'distance_km', 'passenger_count', 'hour'])
        base_fare = model.predict(input_df)[0]

        # Ride Multipliers
        vehicle_type = veh_widget.value
        fare_mod = base_fare * traffic_mult

        if vehicle_type == "Activa": fare_mod *= 0.6
        elif vehicle_type == "Bike": fare_mod *= 0.7
        elif vehicle_type == "Auto Rickshaw": fare_mod *= 0.85
        elif vehicle_type == "Car": fare_mod *= 1.25

        fare_mod += (bags_widget.value * 15)
        fare_final = max(40, round(fare_mod))

        result_html = f"""
        <div class="card" style="border-left: 6px solid #000000; color: #1a1a1a;">
            <h2 style="margin:0 0 10px 0;">💰 Fare Confirmed: ₹{fare_final}</h2>
            <p style="margin:4px 0;"><b>Route:</b> {p_name} ➔ {d_name}</p>
            <p style="margin:4px 0;"><b>Est. Ride Time:</b> {est_time} mins | <b>Distance:</b> {dist:.2f} km</p>
            <p style="margin:4px 0;"><b>Traffic:</b> {traffic} ({traffic_mult}x multiplier applied)</p>
        </div>
        """
        display(HTML(result_html))

        # Show Folium Map
        map_html = show_map(p_coord, d_coord, p_name, d_name)
        display(HTML(map_html))

btn_predict.on_click(on_predict_clicked)

def render_predict():
    with out_predict:
        clear_output()
        display(HTML(f"""{style_header}
        <div class="app-header">
            <h2 style="margin:0;">📍 Request Uber Ride</h2>
            <p style="margin:5px 0 0 0; opacity:0.8;">Enter pickup and drop locations below</p>
        </div>
        """))
        display(widgets.VBox([
            widgets.HBox([p_dd, p_txt]),
            widgets.HBox([d_dd, d_txt]),
            widgets.HBox([pax_widget, hr_widget]),
            widgets.HBox([bags_widget, veh_widget]),
            btn_predict,
            out_predict_box
        ]))

# ------------------ 2. COMPARE RIDES TAB ------------------
btn_compare = widgets.Button(description="⚖️ COMPARE ALL TRANSIT MODES", button_style='primary', layout={'width': '92%', 'height': '40px'})
out_comparison_box = widgets.Output()

def on_compare_clicked(b):
    with out_comparison_box:
        clear_output()
        p_name, p_coord = get_location(p_dd, p_txt)
        d_name, d_coord = get_location(d_dd, d_txt)

        if not p_coord or not d_coord:
            display(HTML("<p style='color:red;'>❌ Set your trip parameters in the Booking tab first.</p>"))
            return

        dist = ((d_coord[0]-p_coord[0])**2 + (d_coord[1]-p_coord[1])**2)**0.5 * 111
        _, _, _, traffic_mult, est_time = analyze_ride_conditions(hr_widget.value, dist)

        p_code = le_pickup.transform([p_name])[0]
        d_code = le_dropoff.transform([d_name])[0]
        input_df = pd.DataFrame([[p_code, d_code, dist, 1, hr_widget.value]],
                                columns=['pickup_code', 'dropoff_code', 'distance_km', 'passenger_count', 'hour'])
        base_fare = model.predict(input_df)[0]

        vehicles = ['Bike', 'Activa', 'Auto Rickshaw', 'Car']
        mults = [0.7, 0.6, 0.85, 1.25]
        time_factors = [0.75, 0.8, 0.9, 1.1]

        rows = ""
        for v, m, tf in zip(vehicles, mults, time_factors):
            fare_est = max(40, round(base_fare * traffic_mult * m))
            dur = max(4, round(est_time * tf))
            rows += f"""
            <tr>
                <td><b>{v}</b></td>
                <td style='color:#2b6cb0; font-weight:bold;'>₹{fare_est}</td>
                <td>{dur} mins</td>
                <td>{dist:.1f} km</td>
            </tr>
            """

        table_html = f"""
        <div class="card" style="color: #1a1a1a;">
            <h3 style="margin-top:0;">⚖️ Available Ride Alternatives</h3>
            <table class="tbl">
                <thead>
                    <tr>
                        <th>Vehicle Profile</th>
                        <th>Est. Price</th>
                        <th>Arrival Time</th>
                        <th>Distance</th>
                    </tr>
                </thead>
                <tbody>
                    {rows}
                </tbody>
            </table>
        </div>
        """
        display(HTML(table_html))

btn_compare.on_click(on_compare_clicked)

def render_compare():
    with out_compare:
        clear_output()
        display(HTML(f"""{style_header}
        <div class="app-header">
            <h2 style="margin:0;">⚖️ Compare Rides</h2>
            <p style="margin:5px 0 0 0; opacity:0.8;">Compare rates, classes and durations live</p>
        </div>
        """))
        display(widgets.VBox([
            btn_compare,
            out_comparison_box
        ]))

# ------------------ 3. SAFETY TAB ------------------
sos_logged = widgets.Output()

def trigger_sos(b):
    with sos_logged:
        clear_output()
        now = datetime.datetime.now().strftime('%H:%M:%S')
        alert_box = f"""
        <div style="background-color:#fff5f5; border-left: 6px solid #e53e3e; padding:15px; border-radius:8px; margin-top:15px; color:#1a1a1a;">
            <h4 style="color:#e53e3e; margin:0 0 5px 0;">🚨 SECURE SOS DISPATCHED</h4>
            <p style="margin:0; font-size:13px;">Live tracking coordinates and vehicle ID successfully sent to Amritsar Central Emergency Dispatch at <b>{now}</b>.</p>
        </div>
        """
        display(HTML(alert_box))

btn_sos = widgets.Button(description="🚨 IN-APP EMERGENCY SOS", button_style='danger', layout={'width': '92%', 'height':'45px'})
btn_sos.on_click(trigger_sos)

def render_safety():
    with out_safety:
        clear_output()
        display(HTML(f"""{style_header}
        <div class="app-header" style="background: #4a154b;">
            <h2 style="margin:0;">🛡️ Women & Safety Protocol</h2>
            <p style="margin:5px 0 0 0; opacity:0.8;">Secure transit shield and real-time support</p>
        </div>
        <div class="card" style="color: #1a1a1a;">
            <h4 style="margin:0 0 8px 0; color:#4a154b;">🔒 Safe Ride Verification PIN</h4>
            <p style="margin:0 0 10px 0; font-size:13px; color:#4a5568;">Provide this PIN to the driver before embarking to activate end-to-end GPS protection.</p>
            <h1 style="color:#4a154b; background-color:#f7ebf6; padding: 10px 20px; display:inline-block; border-radius:8px; margin:0; font-family: monospace; letter-spacing: 2px;">5849</h1>
        </div>
        <div class="card" style="color: #1a1a1a;">
            <h4 style="margin:0 0 5px 0;">👤 Match Status</h4>
            <p style="margin:2px 0; font-size:13px;"><b>Assigned Driver:</b> Simranjeet Kaur (Verified Woman Partner)</p>
            <p style="margin:2px 0; font-size:13px;"><b>Reg Number:</b> PB-02-CD-5928</p>
        </div>
        """))
        display(widgets.VBox([
            widgets.HTML("<p style='font-size:13px; color:#718096;'>Tap below in case of any emergency or unauthorized route deviations.</p>"),
            btn_sos,
            sos_logged
        ]))

# --- App Tab Workspace Initialization ---
def on_tab_changed(change):
    idx = change['new']
    if idx == 0: render_predict()
    elif idx == 1: render_compare()
    elif idx == 2: render_safety()

tab = widgets.Tab()
tab.children = [out_predict, out_compare, out_safety]
tab.set_title(0, '🚗 Book Ride')
tab.set_title(1, '⚖️ Compare Rides')
tab.set_title(2, '🛡️ Women Safety')

tab.observe(on_tab_changed, names='selected_index')

# Trigger Initial Tab View
render_predict()


## Display UI

In [25]:
# Removed raw draft inputs to avoid screen clutter and focus entirely on the main Uber interface below.

### 🚀 Run the Smart Ride Application
Execute the cell below to load the complete interactive dashboard. You can toggle between different tabs to test out fare estimation, ride comparisons, real-time analytics graphs, simulated SOS safety features, and trip histories.

In [32]:
# Initialize rendering for the active default tab layout
render_predict()

# Render clean integrated App Layout header block
display(HTML("""
<div style='background-color:#09091a; padding:18px; border-radius:12px; margin-bottom:15px; text-align:center; font-family:\'Segoe UI\', sans-serif; box-shadow: 0 4px 15px rgba(0,0,0,0.1);'>
    <h1 style='color:#ffffff; margin:0; font-size:24px;'>🚗 Uber Amritsar Multi-Modal Portal</h1>
    <p style='color:#94a3b8; margin:5px 0 0 0; font-size:13px;'>Integrated Machine Learning Front-End Dashboard</p>
</div>
"""))

# Present the main app workspace containing the tabs
display(tab)


### 💾 Step 1: Export Trained ML Model & Preprocessing Encoders
We serialize the trained Random Forest Regressor and our `LabelEncoder` objects so the Streamlit application can load them instantly without retraining.

In [ ]:
import pickle

# Save the model and encoders
with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('encoders.pkl', 'wb') as f:
    pickle.dump({
        'le_pickup': le_pickup,
        'le_dropoff': le_dropoff,
        'location_coords': location_coords
    }, f)

print("✅ Core artifacts serialized successfully!")

### 💻 Step 2: Create Streamlit Frontend Application (`app.py`)
Here, we write the full interactive Python-based web app. It uses custom CSS styles to produce a beautiful, responsive dark-themed dashboard modeled directly after Uber's user experience guidelines.

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import pickle
import datetime

# Set page title and layout
st.set_page_config(
    page_title="Uber Amritsar Fare Predictor",
    page_icon="🚗",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Inject Custom Sleek Uber Dark/Light CSS
st.markdown("""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;600;700&display=swap');
    html, body, [class*="css"] {
        font-family: 'Inter', sans-serif;
    }
    .main {
        background-color: #0a0a0a;
        color: #ffffff;
    }
    .uber-header {
        background-color: #000000;
        padding: 30px;
        border-radius: 12px;
        border: 1px solid #1a1a1a;
        text-align: center;
        margin-bottom: 25px;
        box-shadow: 0 4px 20px rgba(0,0,0,0.4);
    }
    .uber-card {
        background-color: #111111;
        border: 1px solid #222222;
        border-radius: 12px;
        padding: 24px;
        margin-bottom: 20px;
        box-shadow: 0 4px 12px rgba(0,0,0,0.3);
    }
    .stButton>button {
        background-color: #ffffff !important;
        color: #000000 !important;
        font-weight: 600 !important;
        border-radius: 8px !important;
        border: none !important;
        padding: 10px 24px !important;
        width: 100%;
        transition: all 0.3s ease;
    }
    .stButton>button:hover {
        background-color: #e0e0e0 !important;
        transform: translateY(-2px);
    }
    .reset-btn>button {
        background-color: #1c1c1c !important;
        color: #ffffff !important;
        border: 1px solid #333333 !important;
    }
    .reset-btn>button:hover {
        background-color: #2c2c2c !important;
    }
    .metric-num {
        font-size: 3rem;
        font-weight: 700;
        color: #ffffff;
        margin: 10px 0;
    }
</style>
""", unsafe_allow_html=True)

# Helper to load models robustly
@st.cache_resource
def load_ml_assets():
    try:
        with open('model.pkl', 'rb') as m_file:
            model = pickle.load(m_file)
        with open('encoders.pkl', 'rb') as e_file:
            assets = pickle.load(e_file)
        return model, assets['le_pickup'], assets['le_dropoff'], assets['location_coords']
    except Exception as e:
        st.error(f"Failed to load serialized ML assets: {e}")
        return None, None, None, None

model, le_pickup, le_dropoff, location_coords = load_ml_assets()

# Session State for prediction history
if 'history' not in st.session_state:
    st.session_state.history = []

# Main Title Header
st.markdown("""
<div class="uber-header">
    <h1 style="color: white; margin: 0; font-size: 2.5rem; font-weight:700;">🚗 Uber Amritsar</h1>
    <p style="color: #888888; margin: 10px 0 0 0; font-size: 1.1rem;">Machine Learning Multi-Modal Fare Predictor</p>
</div>
""", unsafe_allow_html=True)

# Main Columns Layout
col1, col2 = st.columns([3, 2])

with col1:
    st.markdown('<div class="uber-card">', unsafe_allow_html=True)
    st.markdown('### 📍 Configure Ride Parameters')

    # Pick drop downs from current coordinates dictionary
    locations = sorted(list(location_coords.keys()))

    c_p1, c_p2 = st.columns(2)
    with c_p1:
        pickup_selection = st.selectbox("Pickup Station:", locations, index=locations.index("Golden Temple") if "Golden Temple" in locations else 0)
    with c_p2:
        dropoff_selection = st.selectbox("Dropoff Destination:", locations, index=locations.index("Airport") if "Airport" in locations else 1)

    st.markdown("**Or Enter Customized GPS Coordinates Below:**")
    c_c1, c_c2 = st.columns(2)
    with c_c1:
        pickup_lat = st.number_input("Pickup Latitude:", value=location_coords[pickup_selection][0], format="%.6f")
        pickup_lng = st.number_input("Pickup Longitude:", value=location_coords[pickup_selection][1], format="%.6f")
    with c_c2:
        dropoff_lat = st.number_input("Dropoff Latitude:", value=location_coords[dropoff_selection][0], format="%.6f")
        dropoff_lng = st.number_input("Dropoff Longitude:", value=location_coords[dropoff_selection][1], format="%.6f")

    c_o1, c_c3, c_c4 = st.columns(3)
    with c_o1:
        vehicle_selection = st.selectbox("Ride Type:", ['Car', 'Bike', 'Activa', 'Auto Rickshaw'])
    with c_c3:
        passenger_count = st.number_input("Passengers:", min_value=1, max_value=4, value=1)
    with c_c4:
        bags_count = st.number_input("Luggage Bags:", min_value=0, max_value=5, value=0)

    ride_time = st.time_input("Ride Time:", datetime.datetime.now().time())

    # Action buttons row
    st.markdown('<br/>', unsafe_allow_html=True)
    b_col1, b_col2, b_col3 = st.columns([2, 1, 1])

    with b_col1:
        btn_calc = st.button("⚡ BOOK & PREDICT FARE")
    with b_col2:
        btn_sample = st.button("🧪 SAMPLE INPUT")
    with b_col3:
        st.markdown('<div class="reset-btn">', unsafe_allow_html=True)
        btn_reset = st.button("🧹 RESET")
        st.markdown('</div>', unsafe_allow_html=True)

    # Trigger events
    if btn_sample:
        st.info("Sample Golden Temple to Airport inputs pre-filled above.")

    if btn_reset:
        st.session_state.history = []
        st.success("Form session successfully reset!")
        st.rerun()

    st.markdown('</div>', unsafe_allow_html=True)

with col2:
    st.markdown('<div class="uber-card" style="min-height: 520px;">', unsafe_allow_html=True)
    st.markdown('### 💰 Calculated Booking Summary')

    if btn_calc and model is not None:
        with st.spinner("Calculating best ML fare projections..."):
            # Calculate precise Distance approximation
            dist_km = ((dropoff_lat - pickup_lat)**2 + (dropoff_lng - pickup_lng)**2)**0.5 * 111
            hr = ride_time.hour

            try:
                p_code = le_pickup.transform([pickup_selection])[0]
                d_code = le_dropoff.transform([dropoff_selection])[0]
            except Exception as e:
                # Graceful fallback decoding if custom coordinate search was specified
                p_code = 0
                d_code = 1

            input_df = pd.DataFrame([[p_code, d_code, dist_km, passenger_count, hr]],
                                    columns=['pickup_code', 'dropoff_code', 'distance_km', 'passenger_count', 'hour'])

            base_fare = model.predict(input_df)[0]

            # Apply business modifiers
            fare_mod = base_fare
            if vehicle_selection == "Activa": fare_mod *= 0.6
            elif vehicle_selection == "Bike": fare_mod *= 0.7
            elif vehicle_selection == "Auto Rickshaw": fare_mod *= 0.85
            elif vehicle_selection == "Car": fare_mod *= 1.25

            # Bags
            fare_mod += (bags_count * 15)
            final_fare = max(40, round(fare_mod))

            st.markdown(f"""
            <div style="text-align: center;">
                <p style="color: #888888; font-size: 1.1rem; margin-bottom: 0;">Est. Ride Fare</p>
                <div class="metric-num">₹{final_fare}</div>
            </div>
            """, unsafe_allow_html=True)

            st.markdown("***")
            st.markdown(f"📊 **Trip Details:**")
            st.markdown(f"- 🗺️ **Distance**: {dist_km:.2f} km")
            st.markdown(f"- ⏱️ **Pickup Hour**: {hr}:00 hrs")
            st.markdown(f"- 👥 **Capacity**: {passenger_count} Passenger(s)")
            st.markdown(f"- 🎒 **Baggage**: {bags_count} Bag(s)")

            # Save calculation to session history
            st.session_state.history.append({
                'time': datetime.datetime.now().strftime("%H:%M:%S"),
                'route': f"{pickup_selection} ➔ {dropoff_selection}",
                'vehicle': vehicle_selection,
                'fare': f"₹{final_fare}"
            })
    else:
        st.write("Waiting for user input... Fill route parameters and click predict to calculate ML dispatch fare rates.")
    st.markdown('</div>', unsafe_allow_html=True)

# Render Session Prediction History
st.markdown('<div class="uber-card">', unsafe_allow_html=True)
st.markdown('### 📜 Current Session Prediction History')
if st.session_state.history:
    hist_df = pd.DataFrame(st.session_state.history)
    st.dataframe(hist_df, use_container_width=True)
else:
    st.info("No predictions recorded in this active web portal session yet.")
st.markdown('</div>', unsafe_allow_html=True)
"""

### 📦 Step 3: Write Requirements & Installation Files (`requirements.txt` & `README.md`)
These deployment files allow users to replicate the environment on any server machine or deploy seamlessly to **Streamlit Community Cloud**.

In [ ]:
%%writefile requirements.txt
streamlit>=1.22.0
pandas>=1.3.0
numpy>=1.20.0
scikit-learn>=1.0.0
openpyxl>=3.0.0

In [ ]:
%%writefile README.md
# Uber Amritsar Fare Prediction Web App

This is a clean, production-grade Streamlit web application that serves predictions from our high-accuracy **Random Forest Regressor** trained on the Amritsar Uber Excel dataset.

### 🚀 Quick Start / Local Execution
To run this app on your local computer:

1. Ensure you have the ML models and encoders exported in the root directory:
   - `model.pkl`
   - `encoders.pkl`

2. Install requirements:
   ```bash
   pip install -r requirements.txt
   ```

3. Launch the web app:
   ```bash
   streamlit run app.py
   ```

### ☁️ Deployment Instructions
To deploy this globally on **Streamlit Community Cloud**:
1. Push `app.py`, `requirements.txt`, `model.pkl`, and `encoders.pkl` to a GitHub repository.
2. Login to [Streamlit Community Cloud](https://share.streamlit.io/).
3. Select "New App", bind your repository, and set `app.py` as the main entry point.
4. Click **Deploy!**

In [7]:
import sys
print(sys.executable)
print(sys.version)

C:\Users\shqqq\Downloads\Amritsar_Uber_Project\.venv\Scripts\python.exe
3.13.5 (tags/v3.13.5:6cb20a2, Jun 11 2025, 16:15:46) [MSC v.1943 64 bit (AMD64)]


In [8]:
import pandas as pd
import numpy as np
import sklearn
import matplotlib
import seaborn
import folium
import geopy
import ipywidgets

print("All required libraries imported successfully!")

All required libraries imported successfully!


In [9]:
import os

print("Current folder:", os.getcwd())
print("Files in folder:", os.listdir())

Current folder: C:\Users\shqqq\Downloads\Amritsar_Uber_Project
Files in folder: ['.ipynb_checkpoints', '.venv', 'amritsar_uber_dataset.csv', 'uber_fare_prediction.ipynb.ipynb']


In [10]:
df = pd.read_csv("amritsar_uber_dataset.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)
display(df.head())

Dataset loaded successfully!
Shape: (49310, 21)


,pickup_location,dropoff_location,pickup_latitude,pickup_longitude,dropoff_latitude,dropoff_longitude,pickup_datetime,distance_km,hour,day_of_week,...,is_pilgrimage,is_airport,is_border,is_market,is_night,is_weekend,rush_hour,fare_amount,pickup_enc,dropoff_enc
0,Bharawan Da Dhaba,Kapurthala Road,31.624167,74.780833,31.660000,74.880000,2024-01-01 00:00:00,10.198209,0,0,...,False,False,False,True,True,False,False,286.374108,4,19
1,Mall of Amritsar,GNDU University,31.652222,74.850000,31.663889,74.964444,2024-01-01 00:10:00,10.909378,0,0,...,False,False,False,False,True,False,False,294.223887,24,12
2,Amritsar Bus Stand,Durgiana Temple,31.642222,74.845833,31.616667,74.850000,2024-01-01 00:20:00,2.868843,0,0,...,False,False,False,False,True,False,False,105.186070,1,10
3,Guru Nanak Dev Hospital,Majitha Road,31.652778,74.788889,31.650000,74.900000,2024-01-01 00:30:00,10.521793,0,0,...,False,False,False,True,True,False,False,298.980576,14,23
4,Khalsa College,GNDU University,31.628056,74.860278,31.663889,74.964444,2024-01-01 00:40:00,10.635051,0,0,...,False,False,False,False,True,False,False,285.636201,21,12
